# YOROSIX Cooperation Dataset - Robot Cooperation Example

This notebook demonstrates how to use the YOROSIX Cooperation Dataset to train robots for natural team coordination.

## Use Case

Train multi-robot systems to:
- Identify high-cooperation team members (using ripple scores)
- Form effective robot teams based on cooperation patterns
- Predict cooperation propagation in robot networks
- Develop social intelligence for embodied AI

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# Load the YOROSIX Cooperation Dataset
nodes = pd.read_csv('../data/ml_node_features_v6_3.csv')
edges = pd.read_csv('../data/ml_edge_list_v6_3.csv')
labels = pd.read_csv('../data/ml_node_label_dataset_v6_3.csv')
scores = pd.read_csv('../data/bdf_f1pple_score_v0_3.csv')
predictions = pd.read_csv('../data/gnn_expand_prob_v6_3.csv')

print(f'Nodes: {len(nodes)}')
print(f'Edges: {len(edges)}')
print(f'Cooperation-ready nodes: {labels["label_node_expand"].sum()}')
print(f'High-ripple nodes (score > 0.9): {(scores["f1pple_score"] > 0.9).sum()}')

## 2. Build the Cooperation Network

In [ ]:
G = nx.Graph()

for _, row in edges.iterrows():
    G.add_edge(int(row.iloc[0]), int(row.iloc[1]), 
               weight=float(row.iloc[2]) if len(row) > 2 else 1.0)

print(f'Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'Average degree: {2 * G.number_of_edges() / G.number_of_nodes():.2f}')
print(f'Density: {nx.density(G):.4f}')

## 3. Select Optimal Robot Team

Use ripple scores to identify the most cooperative team members for robot coordination training.

In [ ]:
# Get high-cooperation nodes (ripple score > 0.9)
high_coop = scores[scores['f1pple_score'] > 0.9].sort_values('f1pple_score', ascending=False)

# Select top 5 as the optimal robot team
robot_team = high_coop.head(5)['member_id'].tolist()

print('Optimal Robot Team:')
for i, node_id in enumerate(robot_team, 1):
    score = scores[scores['member_id'] == node_id]['f1pple_score'].values[0]
    print(f'  Robot {i}: Node {node_id} (ripple score: {score:.4f})')

## 4. Visualize the Network

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

pos = nx.spring_layout(G, k=0.5, iterations=50, seed=42)

# Color nodes by role
node_colors = []
for node in G.nodes():
    if node in robot_team:
        node_colors.append('red')
    elif node in high_coop['member_id'].tolist():
        node_colors.append('orange')
    else:
        node_colors.append('lightblue')

node_sizes = [200 if n in robot_team else 50 for n in G.nodes()]

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.8, ax=ax)
nx.draw_networkx_edges(G, pos, alpha=0.1, width=0.3, ax=ax)

ax.set_title('YOROSIX Cooperation Network\nRed: Robot Team | Orange: High-Cooperation | Blue: Others', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Predict Cooperation Propagation

Use GNN predictions to identify which nodes are likely to cooperate when the robot team initiates an action.

In [ ]:
# Get neighbors of the robot team
robot_neighbors = set()
for robot in robot_team:
    if robot in G.nodes():
        robot_neighbors.update(G.neighbors(robot))

print(f'Robot team has {len(robot_neighbors)} direct neighbors in the network')

# Get prediction probabilities for these neighbors
neighbor_predictions = predictions[predictions['member_id'].isin(robot_neighbors)]
neighbor_predictions = neighbor_predictions.sort_values(neighbor_predictions.columns[1], ascending=False)

print('\nTop 10 nodes likely to cooperate with robot team:')
print(neighbor_predictions.head(10))

## Conclusion

This example demonstrates how YOROSIX cooperation data can be used to:

1. **Identify optimal team members** based on cooperation history
2. **Predict cooperation propagation** using GNN models
3. **Design robot coordination strategies** grounded in a synthetic model of human cooperation (see the README: this dataset is synthetic)

## Next Steps

- Apply this data to train multi-agent reinforcement learning models
- Integrate with robot simulators (e.g., Isaac Sim, Gazebo)
- Build cooperation prediction APIs for real-time robot coordination

## Commercial Use

For commercial applications, contact: kazutaka.hayashi@yorosix.com